DDL design notes for `adwm_wh.gold.FactSales`.

Intended grain:

* One row per sales order line
* Proposed business grain is `SalesOrderNumber` + `SalesOrderLineNumber`
* Grain uniqueness should still be enforced in load logic because Databricks key constraints are informational

Platform notes:

* Delta tables do not use traditional indexes, so `CLUSTER BY` is used for common filter columns instead
* Currency-style values should use `DECIMAL(19,4)`
* `ShipDateKey` may need to be nullable or mapped to an unknown date key when orders have not shipped
* `TaxAmt` and `Freight` are commonly header-level measures, so line-level loading should confirm whether allocation is required to avoid double counting

Notebook contents:

* Schema creation for the gold layer
* Proposed fact table DDL with informational keys and clustering

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adwm_wh.gold;

CREATE TABLE IF NOT EXISTS adwm_wh.gold.FactSales (
    SalesKey              BIGINT GENERATED ALWAYS AS IDENTITY,

    OrderDateKey          INT NOT NULL,
    DueDateKey            INT NOT NULL,
    ShipDateKey           INT,
    CustomerKey           INT NOT NULL,
    ProductKey            INT NOT NULL,
    EmployeeKey           INT NOT NULL,

    SalesOrderNumber      STRING NOT NULL,
    SalesOrderLineNumber  SMALLINT NOT NULL,

    OrderQty              SMALLINT NOT NULL,
    UnitPrice             DECIMAL(19,4) NOT NULL,
    UnitPriceDiscount     DECIMAL(19,4) NOT NULL,
    LineTotal             DECIMAL(19,4) NOT NULL,
    TaxAmt                DECIMAL(19,4),
    Freight               DECIMAL(19,4),

    LoadedDate            TIMESTAMP NOT NULL DEFAULT current_timestamp(),

    CONSTRAINT pk_FactSales PRIMARY KEY (SalesKey),
    CONSTRAINT fk_FactSales_OrderDate FOREIGN KEY (OrderDateKey) REFERENCES dim.DimDate(DateKey),
    CONSTRAINT fk_FactSales_DueDate   FOREIGN KEY (DueDateKey)   REFERENCES dim.DimDate(DateKey),
    CONSTRAINT fk_FactSales_ShipDate  FOREIGN KEY (ShipDateKey)  REFERENCES dim.DimDate(DateKey),
    CONSTRAINT fk_FactSales_Customer  FOREIGN KEY (CustomerKey)  REFERENCES dim.DimCustomer(CustomerKey),
    CONSTRAINT fk_FactSales_Product   FOREIGN KEY (ProductKey)   REFERENCES dim.DimProduct(ProductKey),
    CONSTRAINT fk_FactSales_Employee  FOREIGN KEY (EmployeeKey)  REFERENCES dim.DimEmployee(EmployeeKey)
)
USING DELTA
CLUSTER BY (OrderDateKey, CustomerKey, ProductKey);